# Dim_MotivosRejeicao

In [ ]:
#Parameter
run_id = ""

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print(f"{'='*80}")
print("CONSTRUÇÃO DA DIMENSÃO: MOTIVOS DE REJEIÇÃO")
print(f"{'='*80}\n")

df_motivos = spark.read.table("slv.motivosreprovacao")

df_base = df_motivos.filter(
    F.col("motivo_rejeicao").isNotNull()
)

df_base = df_base.filter(
    ~F.upper(F.trim(F.col("motivo_rejeicao"))).isin("NAN", "NULL", "", "NÃO DEFINIDO")
)

window_unique = Window.partitionBy(
    F.upper(F.trim(F.col("motivo_rejeicao")))
).orderBy(
    F.col("sindrome_rejeicao").asc_nulls_last()
)

df_unique = df_base.withColumn(
    "rn",
    F.row_number().over(window_unique)
).filter(
    F.col("rn") == 1
).drop("rn")


# 3.1 REGRA ESPECÍFICA PARA HIPERTERMIA
df_unique = df_unique.withColumn(
    "sindrome_rejeicao",
    F.when(
        F.trim(F.col("motivo_rejeicao")) == "Hipertermia",
        F.lit("Infecção Sistémica/Toxémia/Doença generalizada")
    ).otherwise(F.col("sindrome_rejeicao"))
)

motivo_cols = ["motivo_rejeicao"]

all_null_cond_mot = F.lit(True)
for c in motivo_cols:
    all_null_cond_mot = all_null_cond_mot & F.col(c).isNull()

df_dim_motivo = df_unique.select(
    F.when(all_null_cond_mot, F.lit("MOTIVO_UNDEFINED"))
     .otherwise(
         F.concat(
             F.lit("SK_MOTIVO_"),   #md5 é um hash criptográfico determinístico
             F.md5(
                 F.concat_ws(
                     "|",
                     *[F.coalesce(F.upper(F.trim(F.col(c))), F.lit("N/A")) for c in motivo_cols]
                 )
             )
         )
     ).alias("SK_Motivo_Rejeicao"),

    F.coalesce(F.trim(F.col("motivo_rejeicao")), F.lit("Não Definido")).alias("motivo_rejeicao"),
    F.coalesce(F.trim(F.col("sindrome_rejeicao")), F.lit("Não Definido")).alias("sindrome_rejeicao"),

    "meta_source_file",
    "meta_source_file_date",
    F.current_timestamp().alias("audit_gold_refresh_timestamp")
)

spark.sql("CREATE SCHEMA IF NOT EXISTS gld")

df_dim_motivo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gld.dim_motivorejeicao")

print("Tabela gld.dim_motivo_rejeicao criada com sucesso!")

StatementMeta(, a2977643-164a-4a5b-9d0a-089819b964de, 3, Finished, Available, Finished, False)

CONSTRUÇÃO DA DIMENSÃO: MOTIVOS DE REJEIÇÃO

Tabela gld.dim_motivo_reprovacoes criada com sucesso!


### Criação de tabela de auditoria de parametro run_id do pipeline

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F



# VALIDAR RUN ID RECEBIDO DO PIPELINE

if run_id is None or str(run_id).strip() == "":
    raise ValueError(
        "O parâmetro 'run_id' não foi recebido do pipeline. "
        "Confirma se a atividade Notebook está configurada com "
        "run_id = @pipeline().parameters.p_run_id"
    )

run_id = str(run_id).strip()

print(f"Run ID recebido do pipeline: {run_id}")



# CALCULAR MÉTRICAS ENTRE A VERSÃO ATUAL E A ANTERIOR

def calcular_metricas_delta(
    table_name,
    chaves,
    colunas_ignorar=None
):
    colunas_ignorar = colunas_ignorar or []

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"A tabela '{table_name}' não existe."
        )

    delta_table = DeltaTable.forName(
        spark,
        table_name
    )

    # Apenas as duas versões mais recentes
    historico = (
        delta_table.history(2)
        .select(
            "version",
            "timestamp",
            "operation"
        )
        .orderBy(
            F.col("version").desc()
        )
    )

    versoes = [
        row["version"]
        for row in historico
        .select("version")
        .collect()
    ]

    if not versoes:
        raise ValueError(
            f"Não existe histórico Delta para '{table_name}'."
        )

    versao_atual = versoes[0]

    df_atual = (
        spark.read
        .format("delta")
        .option(
            "versionAsOf",
            versao_atual
        )
        .table(table_name)
    )

    total_rows = df_atual.count()

    # Primeira versão da tabela
    if len(versoes) == 1:
        return {
            "current_version": versao_atual,
            "previous_version": None,
            "total_rows": total_rows,
            "rows_added": total_rows,
            "rows_updated": 0,
            "rows_deleted": 0
        }

    versao_anterior = versoes[1]

    df_anterior = (
        spark.read
        .format("delta")
        .option(
            "versionAsOf",
            versao_anterior
        )
        .table(table_name)
    )

    # Validar chaves
    for chave in chaves:
        if chave not in df_atual.columns:
            raise ValueError(
                f"A chave '{chave}' não existe na versão atual "
                f"de '{table_name}'."
            )

        if chave not in df_anterior.columns:
            raise ValueError(
                f"A chave '{chave}' não existe na versão anterior "
                f"de '{table_name}'."
            )

    chaves_atuais = (
        df_atual
        .select(*chaves)
        .distinct()
    )

    chaves_anteriores = (
        df_anterior
        .select(*chaves)
        .distinct()
    )

    # Linhas adicionadas
    rows_added = (
        chaves_atuais
        .join(
            chaves_anteriores,
            on=chaves,
            how="left_anti"
        )
        .count()
    )

    # Linhas eliminadas
    rows_deleted = (
        chaves_anteriores
        .join(
            chaves_atuais,
            on=chaves,
            how="left_anti"
        )
        .count()
    )

    # Colunas a comparar para identificar atualizações reais
    colunas_comparacao = [
        coluna
        for coluna in df_atual.columns
        if coluna in df_anterior.columns
        and coluna not in chaves
        and coluna not in colunas_ignorar
    ]

    atual = df_atual.alias("atual")
    anterior = df_anterior.alias("anterior")

    condicao_join = F.lit(True)

    for chave in chaves:
        condicao_join = (
            condicao_join
            & F.col(f"atual.{chave}")
            .eqNullSafe(
                F.col(f"anterior.{chave}")
            )
        )

    condicao_alteracao = F.lit(False)

    for coluna in colunas_comparacao:
        condicao_alteracao = (
            condicao_alteracao
            | ~F.col(f"atual.{coluna}")
            .eqNullSafe(
                F.col(f"anterior.{coluna}")
            )
        )

    rows_updated = (
        atual
        .join(
            anterior,
            on=condicao_join,
            how="inner"
        )
        .filter(condicao_alteracao)
        .select(
            *[
                F.col(f"atual.{chave}").alias(chave)
                for chave in chaves
            ]
        )
        .distinct()
        .count()
    )

    return {
        "current_version": versao_atual,
        "previous_version": versao_anterior,
        "total_rows": total_rows,
        "rows_added": rows_added,
        "rows_updated": rows_updated,
        "rows_deleted": rows_deleted
    }



# ESCREVER AUDITORIA EM APPEND

def escrever_auditoria_pipeline(
    layer,
    notebook_name,
    table_name,
    metricas,
    run_id,
    status="success"
):
    if run_id is None or str(run_id).strip() == "":
        raise ValueError(
            f"Não é possível auditar '{table_name}': "
            "run_id está vazio."
        )

    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS audit"
    )

    audit_df = (
        spark.createDataFrame([
            Row(
                run_id=str(run_id).strip(),
                layer=str(layer),
                notebook_name=str(notebook_name),
                table_name=str(table_name),
                status=str(status),
                total_rows=int(metricas["total_rows"]),
                rows_added=int(metricas["rows_added"]),
                rows_updated=int(metricas["rows_updated"]),
                rows_deleted=int(metricas["rows_deleted"])
            )
        ])
        .withColumn(
            "audit_timestamp",
            F.current_timestamp()
        )
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(
            "audit.pipeline_audit_log"
        )
    )

    print(
        f"Auditoria registada | "
        f"run_id={run_id} | "
        f"table={table_name} | "
        f"versão anterior={metricas['previous_version']} | "
        f"versão atual={metricas['current_version']} | "
        f"total={metricas['total_rows']} | "
        f"added={metricas['rows_added']} | "
        f"updated={metricas['rows_updated']} | "
        f"deleted={metricas['rows_deleted']}"
    )



# CONFIGURAÇÃO DA DIMENSÃO MOTIVO DE REJEIÇÃO

tabela_motivo = "gld.dim_motivorejeicao"

chave_motivo = [
    "SK_Motivo_Rejeicao"
]



# CALCULAR MÉTRICAS

metricas_motivo = calcular_metricas_delta(
    table_name=tabela_motivo,
    chaves=chave_motivo,
    colunas_ignorar=[
        "audit_gold_refresh_timestamp"
    ]
)

print("Métricas dim_motivorejeicao:")
print(metricas_motivo)



# ESCREVER NA AUDITORIA COM O RUN ID DO PIPELINE

escrever_auditoria_pipeline(
    layer="gold",
    notebook_name="dim_motivorejeicao",
    table_name=tabela_motivo,
    metricas=metricas_motivo,
    run_id=run_id
)

print(
    f"Auditoria de dim_motivorejeicao concluída com sucesso. "
    f"Run ID: {run_id}"
)

## Validação

In [2]:
# df = spark.read.table("gld.dim_motivorejeicao")

# display(df.limit(50))

StatementMeta(, a2977643-164a-4a5b-9d0a-089819b964de, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ba59f1a-b7d4-4ef4-a793-b8a6df6f99e4)